# The target and the forced response

Every member follows the same forcing path by year. Before a model can read ENSO, that shared path has to come out, and "wet" needs
one threshold.


## Overview

1. Name the test members before anything is fit.
2. Remove the forced response and check it with a test that can fail.
3. Define "wet" against one pooled threshold; read the base rate per half.
4. Read the target against the input.

Shared code lives in `lensml.py`; `show()` prints a function's source.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Pythia Foundations: Introduction to Pandas](https://foundations.projectpythia.org/core/pandas/pandas/) | Helpful | the table every chapter reads is a DataFrame |
| [Chapter 1](01-find-the-data.ipynb) | Helpful | the question and the cache |
| Polynomial fits, quantiles, a binomial standard error | Helpful | `np.polyfit`, `Series.quantile`, √(p(1−p)/n) |

- **Time to learn**: about 15 minutes; every cell runs in seconds.
- **System requirements**: the repository's `environment.yml`; no account, no data beyond the committed cache.


## Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lensml import (
    ANOMALIES,
    SEASON,
    apply_transforms,
    cite,
    decile_edges,
    draw_test_members,
    fit_transforms,
    load_cache,
    show,
    show_fig,
    wet_by_bin,
)
from scipy.stats import norm

## Name the test members first

`TEST_FRACTION` of each half's members are drawn from a `RandomState` seeded with `RNG_SEED`, so re-running redraws the same members.
Nothing is fit on them; [chapter 6](06-test-and-justify.ipynb) scores them. They are not a blind test set (see the note).

:::{admonition} Why these test members are not a blind test set
:class: dropdown
During development these same members were scored with the Niño-sign rule, with logistic and boosting at the base-rate
threshold, and, in the companion course, with a small neural network (chapter 6 gives the details), so a result on them is provisional; a fresh test would need data no choice was made on. Chapter 7 uses the same members' SSP3-7.0 winters.
:::


In [ ]:
table, cache_header, access_date = load_cache()
show(draw_test_members)

In [ ]:
test_members = draw_test_members(table)
table["split"] = np.where(
    table["member_id"].isin(test_members), "test", "train"
)
is_train = table["split"].eq("train")
print(f"test members (scored in chapter 6): {test_members}")
print(f"training members: {sorted(table.loc[is_train, 'member_id'].unique())}")

## Remove the forced response

The raw SON Niño 3.4 box mean and the winter precipitation both carry a forced trend (the `corr(…, year)` lines below print it). A degree-`FORCED_DEG` polynomial in year, fit to the *training-member* ensemble mean, is
subtracted from every member, leaving its own departure: the ENSO state, not the decade. Otherwise the models would learn
time-of-century as ENSO skill.

Then the target: `wet` = 1 when detrended precipitation sits above the `WET_QUANTILE` of the pooled *training* members, **one** threshold
for every member. One curve is fit to both forcing halves together; the per-half base-rate check below supports pooling them, though it
cannot see every difference a forcing half could make (chapter 6, limit 6).

:::{admonition} Why one threshold, fit on training members only
:class: dropdown
A per-member quantile would fix each member's base rate by construction, and the per-half check below could never fail. Curve and
threshold are fit on training members and applied to everyone else, as a `StandardScaler` inside a `Pipeline` is (scikit-learn's
*Common pitfalls*, §12.2), so every validation fold in chapters 3–5 refits them.
:::


In [ ]:
show(fit_transforms, apply_transforms)

In [ ]:
TRANSFORMS = fit_transforms(table[is_train])  # fit on ALL training members
table = apply_transforms(table, TRANSFORMS)
WET_THRESHOLD = TRANSFORMS["wet_threshold"]
print(
    f"wet threshold (pooled training members) = {WET_THRESHOLD:+.4f} mm/day "
    f"of detrended {SEASON} precipitation"
)

### The check that can fail

A forced path the curve missed survives as decades whose mean anomaly sits away from zero. The removal **passes** only if every
decade's mean lies within `±DECADE_TOL` standard errors; the `forced-response removal` line gives the verdict and the chance of a false alarm.

:::{admonition} Why not test the correlation with year
:class: dropdown
The correlation of the anomaly with year is zero by construction for any `FORCED_DEG >= 1`, so it could never fail.
:::


In [ ]:
DECADE_TOL = 3.0
d = table[is_train]
decade = (d["winter"] // 10) * 10
for anom, raw in ANOMALIES.items():
    r = d[raw].corr(d["winter"])
    print(
        f"corr({raw:20s}, year) = {r:+.3f}   "
        "(the forced trend in the raw series, training members)"
    )
removal_ok, n_tests = True, 0
for anom in ANOMALIES:
    g = d.groupby(decade)[anom]
    z = g.mean() / (d[anom].std() / np.sqrt(g.size()))
    worst = z.abs().idxmax()
    ok = bool((z.abs() <= DECADE_TOL).all())
    removal_ok &= ok
    n_tests += len(z)
    print(
        f"{anom:12s} decade-mean anomaly in standard errors: worst {z[worst]:+.2f} "
        f"(the {worst}s) · tolerance ±{DECADE_TOL:.0f} → {'PASS' if ok else 'FAIL'}"
    )
false_alarm = 1 - (1 - 2 * norm.sf(DECADE_TOL)) ** n_tests
verdict = (
    "PASS"
    if removal_ok
    else "FAIL — a forced path survived; change FORCED_DEG"
)
print(
    f"forced-response removal: {verdict}   ({n_tests} decade means; chance a "
    f"correct removal trips the tolerance somewhere ≈ {false_alarm:.0%})"
)

```{admonition} Try it: make the check fail
:class: tip
Set `FORCED_DEG = 0` in `lensml.py`, restart the kernel and re-run. Which line flips to `FAIL`, in which decades? Restore
`FORCED_DEG = 3` after.
```

## The base rate, per half

With `WET_QUANTILE = 2/3` the **base rate** sits near one third, so "always dry" scores about two-thirds accuracy, so this cookbook
reports precision and recall (scikit-learn's *Classification metrics* guide) against the base rate. The `per-half base rate` table is
the check before pooling the halves; the cell also prints each training member's input–target correlation.


In [ ]:
train_tbl = table[is_train]
base_rate = train_tbl["wet"].mean()
print(
    f"base rate P(wet), training members = {base_rate:.3f}  "
    f"(n = {len(train_tbl)} member-winters)"
)
print(
    "\nper-half base rate (training members) — free to differ because the "
    "threshold is one for both halves:"
)
print(
    train_tbl.groupby("forcing_variant")["wet"].agg(["mean", "size"]).round(3)
)
print("\nper-half detrended precipitation (training members):")
stats = ["mean", "std", "median"]
print(train_tbl.groupby("forcing_variant")["prect_anom"].agg(stats).round(3))


def corr_in(d):
    return d["nino34_anom"].corr(d["prect_anom"])


per_member_corr = train_tbl.groupby("member_id").apply(corr_in)
print("\nper-member correlation(nino34_anom, prect_anom), training members:")
print(per_member_corr.round(3).to_string())
pos, neg = int((per_member_corr > 0).sum()), int((per_member_corr < 0).sum())
n_m, sd_m = len(per_member_corr), per_member_corr.std(ddof=1)
print(
    f"\nper-member sign count: {pos} positive · {neg} negative of {n_m} "
    f"training members · mean {per_member_corr.mean():+.3f} · SD {sd_m:.3f}"
)
by_half = train_tbl.groupby(["forcing_variant", "member_id"]).apply(corr_in)
half_means = by_half.groupby(level=0).mean()
print(
    "per-half mean correlation: "
    + " · ".join(f"{half} {v:+.3f}" for half, v in half_means.items())
)
t_mean = per_member_corr.mean() / (sd_m / np.sqrt(n_m))
print(
    f"mean per-member correlation in standard errors (SD / √{n_m}): {t_mean:+.2f}"
    f" → {'distinguishable' if abs(t_mean) > 2 else 'indistinguishable'} from zero"
    " at 2 standard errors"
)

In [ ]:
d = table[is_train]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(
    d["prect_anom"],
    bins=30,
    color="#C9A23A",
    edgecolor="#C9A23A",
    linewidth=0.5,
)
axes[0].axvline(
    WET_THRESHOLD, lw=1, c="k", ls="--", label="wet threshold (training)"
)
axes[0].legend()
axes[0].set(
    xlabel=f"{SEASON} Front Range precipitation, forced response removed (mm/day)",
    ylabel="member-winters",
    title="Target distribution (training members)",
)
marks = {
    0: ("#1b9e77", "o", "dry"),
    1: ("#e7298a", "^", "wet"),
}  # colour AND shape
for wet, (c, m, lab) in marks.items():
    dd = d[d.wet == wet]
    axes[1].scatter(
        dd["nino34_anom"],
        dd["prect_anom"],
        s=12,
        alpha=0.6,
        c=c,
        marker=m,
        label=lab,
    )
axes[1].axvline(0, lw=0.8, c="k")
axes[1].legend()
axes[1].set(
    xlabel="autumn Niño 3.4 anomaly, forced response removed (K)",
    ylabel=f"{SEASON} precipitation anomaly (mm/day)",
    title="Input vs target",
)
cite(fig, access_date)
plt.tight_layout()
show_fig(
    fig,
    (
        "Two panels from the training members: a histogram of the winter "
        "precipitation anomaly with the wet threshold marked as a dashed "
        "line, and a scatter of the autumn Niño 3.4 anomaly against the "
        "precipitation anomaly, dry winters as teal circles and wet "
        "winters as magenta triangles."
    ),
)

**Reading the straight-line relation.** NWS Boulder's sign for east of the Divide is positive — warmer autumn Pacific, wetter winter — and, for La Niña, the opposite (wetter) for
the northwest and north-central mountains, part of the box's west column. Its El Niño-wet half is stated mainly for late
winter and spring, of which DJF holds only February; its La Niña-dry half for "the entire cold season", which DJF is part of. Read the `per-member sign count` line: do the members share a
sign? The `standard errors` line says whether the mean is distinguishable from zero. Stagge et al. set the observed context: over a
climatological model, "the ENSO term explained an additional 2% of total variance" in their regional model, a share of
variance across gauges, which they say "should be viewed in the context of extreme spatial and elevation precipitation differences", and their northern dipole model ("North–Wyoming") is bounded
by 41.2° to 45.5° N. By latitude the Front Range, at about 38.5–40.5° N, falls outside the North–Wyoming model and sits mostly between the poles, its northern edge just past 40° N; and in the
northern pole the sign is reversed, with "increased winter precipitation during La Niña events".

## The wet rate by tenth of the input

A correlation sees only a straight line. The cell below prints the wet share in ten equal-count bins of the input, with a binomial
standard error and the distance from the base rate in those errors.


In [ ]:
show(wet_by_bin)
edges = decile_edges(
    train_tbl
)  # chapter 6 applies these same edges to the test members
bins, chi2, p_chi2, dof = wet_by_bin(train_tbl, edges)
print(
    f"wet rate by decile of the autumn Niño 3.4 anomaly, training members "
    f"(base rate {base_rate:.3f}):"
)
print(bins.round(3).to_string())
print(
    f"chi-square test of wet against decile, training members: "
    f"chi2 = {chi2:.1f}, dof = {dof}, p = {p_chi2:.2g}"
)

The `chi-square` line says whether the wet rate differs between bins beyond chance. In the table, the coldest bin (the strongest La Niña
autumns) and the warmest bins sit below the base rate, the middle above it: a shape no straight line follows. One bin's departure can be
luck; chapter 6 applies these edges to the test members.

```{admonition} Try it: may the halves be pooled?
:class: tip
From the per-half tables, argue whether pooling cmip6 and smbb is justified. Then compare the sign you wrote before chapter 1's data cells
with the decile table: which rule — the NWS sign, its reverse, or neither — would the bins reward?
```


## The one input, and the ones not chosen

One input keeps the model readable whole. Any other field is fair only as an **autumn** value, known before the winter; NWS Pueblo's
outlook names PNA and NAO as the patterns with the "highest correlations wet/dry".

:::{admonition} The candidate inputs in the catalogue
:class: dropdown
| Autumn (SON) field | What it stands for | Here |
|---|---|---|
| TS over `NINO34` | the ENSO state | the one input (`FEATURES`) |
| PSL or Z3 over the North Pacific | a stand-in for the autumn PNA state | the strongest next candidate |
| TREFHT over `FRONT_RANGE` | the box's own autumn temperature | weaker: the box's state, not the teleconnection |
| any field averaged over the winter months | the outcome's own season | never: known only after the outcome |
| `forcing_variant` | which half a member belongs to | not an input; the per-half check decides pooling |

A new input means a new box mean in `derive_member_winters` and a rebuild, so it is a costed next change, never a quiet edit to `FEATURES`.
:::


## Summary

Test members were named first. The forced response came out with a curve fit on training members, checked by a test that can fail.
"Wet" is one pooled threshold, so the per-half base rates are a real check. The straight-line relation is weak; the decile table shows
a shape a line misses.

### What's next?

[Chapter 3](03-split-by-member.ipynb) builds member-grouped folds and measures what a member-blind split, or transforms fit before
it, would do to the score.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| Rodgers, K. B. et al. (2021). *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the ensemble (required citation, with the dataset) | CC BY 4.0 |
| scikit-learn User Guide, §12 Common pitfalls and recommended practices; §3.4 Metrics and scoring. https://scikit-learn.org/stable/ | transforms fit on training data only; the base rate | BSD-3-Clause |
| Stagge, J. H. et al. (2023). Orographic amplification of El Niño teleconnections on winter precipitation across the Intermountain West of North America. *Nature Water* 1, 1016–1026. https://doi.org/10.1038/s44221-023-00163-9 | the two poles | CC BY 4.0 |
| NWS Boulder (2020). La Niña and the Upcoming 2020-2021 Winter Season (some slides reused from 2010). https://www.weather.gov/media/bou/LaNinaWinterOutlook_Sept20.pdf | the expected sign and its season | US Government |
| NWS Pueblo (2025). Winter of 2025-2026 Outlook. https://www.weather.gov/media/pub/pdf/SC_SE_CO_Winter_2025_26_Outlook.pdf | PNA and NAO | US Government work |
